# 03. Labeling Networks Without Clutter

Notebook 02 studied a single question — *where to place the nodes* — and treated overlap as a measurable, rendered-space problem. This notebook follows the same workflow for a different and equally common failure mode: **label overlap**.

We use the *Game of Thrones* character network, a weighted social graph dense enough that a "label everything" strategy fails immediately. The path is the same as before:

1. start from a weak but honest baseline
2. add one deliberate design choice at a time
3. **measure** the clutter each version produces, rather than judging by eye alone
4. stop adding labels when the picture stops getting easier to read

**Learning goals**

- see why "label everything" is never the right default on dense node-link diagrams
- prepare a weighted layout *before* thinking about annotation
- measure label overlap in rendered space, just as notebook 02 measured node overlap
- compare weak internal labels with selective external callouts on the same layout and on the same metric

Continue with `04-game-of-thrones-network-case-study.ipynb` for communities, brokerage, matrix views, and a final multi-panel figure.


In [ ]:
from pathlib import Path
import sys
from textwrap import fill

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()
TEXT = apply_teaching_rc(grid=False, font_base=11.5)
FIG = make_figure_size_scale(focus=(7.2, 7.2))

NETWORK_DATA_DIR = tutorials_dir / "04-networks" / "data"

# Restrained styling defaults reused across every figure in the notebook.
# Keeping these in one place makes the figures comparable: every panel uses
# the same node marker and the same edge stroke, so the only thing that
# changes across figures is the labeling strategy.
LINK_WIDTH = 0.7
LINK_ALPHA = 0.6
SOCIAL_EDGE_COLOR = "#C8CED8"
BASE_NODE_STYLE = dict(node_size=26, node_color="white", edgecolors="#111111", linewidths=LINK_WIDTH)
EDGE_STYLE = dict(width=LINK_WIDTH, alpha=LINK_ALPHA, edge_color=SOCIAL_EDGE_COLOR)
CONNECTOR_STYLE = dict(arrowstyle="-", color="#94A3B8", lw=LINK_WIDTH, shrinkA=4, shrinkB=4, alpha=LINK_ALPHA)
LABEL_BBOX = dict(boxstyle="round,pad=0.22", fc="white", ec="#D5DDE7", lw=0.8, alpha=0.97)


def draw_network_base(ax, G, pos, *, highlight_nodes=None, highlight_color=None):
    """Draw the shared network substrate: edges, default nodes, and optional highlights."""
    nx.draw_networkx_edges(G, pos, ax=ax, **EDGE_STYLE)
    nx.draw_networkx_nodes(G, pos, ax=ax, **BASE_NODE_STYLE)
    if highlight_nodes:
        color = highlight_color if highlight_color is not None else lighten_color(DV_PALETTE["gold"], 0.78)
        nx.draw_networkx_nodes(
            G, pos,
            nodelist=list(highlight_nodes),
            node_size=62,
            node_color=[color] * len(highlight_nodes),
            edgecolors="black", linewidths=0.9, ax=ax,
        )
    return ax


## First Use Guide: Shared Drawing Helpers

Every figure in this notebook uses the same three helpers, so we introduce them once and reuse them afterwards. This mirrors the pattern from notebook 02: keep measurement and styling in reusable functions so each drawing cell can focus on the one design choice it is teaching.

- **`fit_network_axes(ax, pos)`** — fit the axes tightly around the layout with a small symmetric padding. Without this, matplotlib's default auto-scaling gives inconsistent margins across figures and the clutter metric drifts between panels.
- **`add_panel_header(ax, title, subtitle)`** — reserve space at the top of the axes for a wrapped title + optional subtitle drawn as figure-level text, then hide the axis spines. This is the same "external header" pattern used in notebook 02 so titles never overlap the drawing area.
- **`draw_network_base(ax, G, pos, highlight_nodes=...)`** — draw edges and default nodes with the shared `EDGE_STYLE` / `BASE_NODE_STYLE` constants, plus an optional emphasis layer for a labeled subset. **All figures in this notebook draw links with the same `width=0.7`, `alpha=0.6` style**, so the only thing that changes across panels is the labeling strategy.

The annotation logic (internal dense labels vs. external callouts) is introduced later, in Step 5, where it is the main teaching point.


In [ ]:
def fit_network_axes(ax, pos, pad=0.08):
    """Fit axes to the layout with symmetric padding."""
    coords = np.asarray(list(pos.values()), dtype=float)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    span = np.maximum(maxs - mins, 1e-6)
    ax.set_xlim(mins[0] - span[0] * pad, maxs[0] + span[0] * pad)
    ax.set_ylim(mins[1] - span[1] * pad, maxs[1] + span[1] * pad)
    return ax


def select_top_nodes(metric, n=8):
    """Return the n nodes with the largest value in the given metric dict."""
    return [node for node, _ in sorted(metric.items(), key=lambda item: (-item[1], item[0]))[:n]]


def add_panel_header(ax, title=None, subtitle=None):
    """Reserve a header above the axes and draw wrapped title/subtitle as figure text."""
    fig = ax.figure
    pos = ax.get_position()
    fig_width, fig_height = fig.get_size_inches()

    def pts_to_fig_y(points):
        return (points / 72.0) / fig_height

    def width_to_chars(width_fraction, chars_per_inch=11):
        panel_width_in = max(fig_width * width_fraction, 2.0)
        return max(28, int(panel_width_in * chars_per_inch))

    wrapped_title = fill(title, width=width_to_chars(pos.width, chars_per_inch=9)) if title else None
    wrapped_subtitle = fill(subtitle, width=width_to_chars(pos.width)) if subtitle else None

    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0
    subtitle_lines = wrapped_subtitle.count("\n") + 1 if wrapped_subtitle else 0

    title_line_pts = TEXT["panel_title"] * 1.08
    subtitle_line_pts = TEXT["annotation"] * 1.22
    top_margin_pts = 2
    gap_pts = 3 if wrapped_title and wrapped_subtitle else 0
    bottom_margin_pts = 4 if (wrapped_title or wrapped_subtitle) else 0

    header_pts = top_margin_pts + bottom_margin_pts
    if wrapped_title:
        header_pts += title_lines * title_line_pts
    if wrapped_subtitle:
        header_pts += gap_pts + subtitle_lines * subtitle_line_pts

    min_header_pts = 18 if (wrapped_title or wrapped_subtitle) else 0
    header_height = min(pts_to_fig_y(max(header_pts, min_header_pts)), pos.height * 0.18)
    new_height = max(pos.height - header_height, pos.height * 0.68)
    ax.set_position([pos.x0, pos.y0, pos.width, new_height])

    header_top_y = pos.y0 + pos.height - pts_to_fig_y(top_margin_pts)
    current_top = header_top_y

    if wrapped_title:
        fig.text(
            pos.x0,
            current_top,
            wrapped_title,
            ha="left",
            va="top",
            fontsize=TEXT["panel_title"],
            color="#17212B",
        )
        current_top -= pts_to_fig_y(title_lines * title_line_pts + gap_pts)

    if wrapped_subtitle:
        fig.text(
            pos.x0,
            current_top,
            wrapped_subtitle,
            ha="left",
            va="top",
            fontsize=TEXT["annotation"],
            color=DV_PALETTE["gray"],
        )

    ax.set_axis_off()
    return ax


## The Case Study: *Game of Thrones* Character Network

Each edge is an interaction between two characters in Book 1, weighted by how often they appear together. The dataset is small enough to draw in a single panel, dense enough that naive labeling fails, and recognisable enough that the labels genuinely carry meaning for the reader.


In [ ]:
got_edge_table = pd.read_csv(NETWORK_DATA_DIR / "got" / "book1.csv")
got_edge_table.head(2)


In [ ]:
got_graph = nx.from_pandas_edgelist(
    got_edge_table,
    source="Source",
    target="Target",
    edge_attr="weight",
    create_using=nx.Graph(),
)

print(f"{got_graph.number_of_nodes()} characters, {got_graph.number_of_edges()} weighted interactions")


## Step 1: A Weak but Honest Baseline, on an Overlap-Free Layout

Start deliberately weak. A first draft answers a single question: what does the raw network look like before we add any analytical emphasis?

One thing is **not** a fair weakness to inherit into the baseline: *node-marker overlap*. Notebook 02 ended on exactly this point — when a hand-tuned Python refinement is overkill, **one call to Graphviz's `neato` engine with `overlap=prism` gives us a layout in which no two markers overlap**. That is the reference layout we reuse for every figure in the rest of the notebook, so every comparison is done on identical geometry and the only thing that changes is the labeling strategy.

This baseline already tells us two useful things:
- the network is dense enough that a default node-link view is difficult to read
- no character, tie, or region stands out — every node looks identical

### First Use Guide: `nx.nx_agraph.graphviz_layout`

A thin wrapper around `pygraphviz`. Given a `networkx` graph it returns a position dictionary computed by the requested Graphviz engine:

```python
pos = nx.nx_agraph.graphviz_layout(G, prog="neato", args="-Goverlap=prism")
```

- `prog="neato"` — force-directed layout suitable for undirected graphs.
- `-Goverlap=prism` — run the **proximity-stress** overlap-removal post-pass. It is more aggressive than `vpsc` and **guarantees that no two node markers overlap** in the returned geometry, which is what we need here. (`vpsc` reduces overlap but does not eliminate it for dense graphs, so it is the wrong tool for a "reference layout" role.)

The same `got_layout` is reused in every figure below, so when the label-clutter metric changes it is purely because the labeling strategy changed.


In [ ]:
# Reuse one overlap-free Graphviz layout in every figure so only the labeling strategy changes.
got_layout = nx.nx_agraph.graphviz_layout(got_graph, prog="neato", args="-Goverlap=prism")


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
draw_network_base(ax, got_graph, got_layout)
fit_network_axes(ax, got_layout, pad=0.06)
add_panel_header(
    ax,
    "Baseline node-link view (Graphviz neato + prism)",
    "No node markers overlap. The plot does not yet tell us which characters or ties matter — that is the job of the next steps.",
)
plt.show()


## Step 2: Measure Label Clutter Before Fixing It

Just as notebook 02 insisted on measuring **node overlap** in rendered space before trying to reduce it, we measure **label overlap** the same way: in points, on the actual figure, after matplotlib has placed every text artist.

### Why rendered-space measurement matters for labels

A label's visual footprint is determined by **font size** and **character count**, not by anything in the graph. The same label set can look fine at 6 × 6 inches and collide everywhere at 4 × 4. So the metric has to be computed after the figure is drawn.

### The metric we use throughout this notebook

For every figure that carries labels, we report:

> **how many of the drawn labels overlap at least one other label**, written as `n_overlap / n_total` (for example `11/18`) and as a percentage.

This is the direct analogue of the touching-neighbour metric from notebook 02, applied to text bounding boxes rather than node disks. A small helper does the work:

1. force matplotlib to draw the figure (`fig.canvas.draw()`) so every text artist has a real bounding box
2. collect each label's bounding box in display (pixel) coordinates via `get_window_extent`
3. count pairwise bounding-box intersections with `Bbox.overlaps`
4. return the fraction of labels that touch at least one neighbour

The metric is honest about what the *reader* sees, because it measures the actual rendered text — not an abstract approximation.


In [ ]:
def measure_label_overlap(fig, ax):
    """Measure label clutter as the fraction of rendered text boxes that overlap another.

    For `ax.annotate(...)` calls, matplotlib's `get_window_extent` combines the text box
    with the arrow patch, so a long connector would spuriously inflate the bbox. We work
    around this by preferring the wrapping `bbox_patch` when present (that patch covers
    only the text), and falling back to the text's own extent otherwise.
    """
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bboxes = []
    for artist in list(ax.texts):
        if not artist.get_visible() or not str(artist.get_text()).strip():
            continue
        patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
        bbox = patch.get_window_extent(renderer=renderer) if patch is not None else artist.get_window_extent(renderer=renderer)
        bboxes.append(bbox)

    n_total = len(bboxes)
    if n_total < 2:
        return 0.0, 0, n_total

    overlapping = [False] * n_total
    for i in range(n_total):
        for j in range(i + 1, n_total):
            if bboxes[i].overlaps(bboxes[j]):
                overlapping[i] = True
                overlapping[j] = True

    n_overlap = int(sum(overlapping))
    return n_overlap / n_total, n_overlap, n_total


def summarize_label_clutter(fig, ax, *, note):
    """Return both the subtitle string and a reusable summary dict for the current labels."""
    frac, n_overlap, n_total = measure_label_overlap(fig, ax)
    subtitle = f"{n_overlap}/{n_total} labels overlap another label ({frac:.0%}). {note}"
    summary = {"n_overlap": n_overlap, "n_total": n_total, "fraction": frac}
    return subtitle, summary


## Step 3: The "Label Everything" Anti-Pattern

To demonstrate why label selection matters, we first deliberately **label every node** with `nx.draw_networkx_labels`. This is almost always a mistake on a dense network, but seeing the failure mode on the metric — not just on the image — is what makes the lesson stick.


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
draw_network_base(ax, got_graph, got_layout)
nx.draw_networkx_labels(
    got_graph, got_layout,
    font_size=TEXT["dense_label"], font_color="#27313B", ax=ax,
)
fit_network_axes(ax, got_layout, pad=0.06)

subtitle, label_all_summary = summarize_label_clutter(
    fig, ax,
    note="Labeling every character drowns the figure in text and destroys any chance of reading structure.",
)
add_panel_header(ax, "Weak choice: label every node", subtitle)
plt.show()

print(f"Label-all clutter: {label_all_summary['n_overlap']}/{label_all_summary['n_total']} labels overlap another ({label_all_summary['fraction']:.0%}).")


## Step 4: Pick a Justified Subset

Good network labeling is **selective and justified**. "Selective" means we pick a small number of labels. "Justified" means the selection rule is explicit, not chosen by eye.

A natural selection rule for a social network is **weighted degree** — the total interaction strength of a character. The most active characters are the ones a reader benefits most from identifying, and the ranking is reproducible.

The next cell computes that ranking explicitly. The figure after it then uses exactly that subset, so students can see the difference between **choosing labels** and **placing labels**.


In [ ]:
weighted_degree = dict(got_graph.degree(weight="weight"))
top_labeled_nodes = select_top_nodes(weighted_degree, n=8)

pd.DataFrame(
    [(node, weighted_degree[node]) for node in top_labeled_nodes],
    columns=["character", "weighted_degree"],
)


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
draw_network_base(ax, got_graph, got_layout, highlight_nodes=top_labeled_nodes)
nx.draw_networkx_labels(
    got_graph, got_layout,
    labels={node: node for node in top_labeled_nodes},
    font_size=TEXT["direct_label"], font_weight="semibold", font_color="#1F2933", ax=ax,
)
fit_network_axes(ax, got_layout, pad=0.06)

subtitle, internal_label_summary = summarize_label_clutter(
    fig, ax,
    note="Only 8 labels, but several still collide where the core is densest.",
)
add_panel_header(ax, "Selective labels, placed internally", subtitle)
plt.show()


## Step 5: Move Labels Outside the Core With External Callouts

If a label cannot fit where it belongs, we **move it outward and connect it back** with a thin leader line. This is the standard solution in professional figures: the reader still knows which node each label refers to, but the text has escaped the crowded core.

### First Use Guide: `ax.annotate(...)` for callouts

`ax.annotate` is matplotlib's tool for text-with-arrow:

- `xy=(x, y)` — the data point the label refers to (here, the node position).
- `xytext=(dx, dy)` with `textcoords="offset points"` — offset in rendered points from `xy`, which is exactly the space we want: labels move by the same amount regardless of the data scale.
- `arrowprops=dict(arrowstyle="-", ...)` — a thin connector line. We pick a simple `"-"` style so the callout does not add visual weight.
- `bbox=dict(boxstyle="round,pad=...", fc="white", ec=...)` — a rounded background behind the text, so the callout is legible when it overlaps edges.

The helper below applies this idea to a list of nodes. Each label is pushed to the **left or right** depending on which side of the layout the node sits on (relative to the centroid), then the vertical positions are adjusted with a minimum gap so labels on the same side do not stack on top of each other.


In [ ]:
def add_external_callout_labels(ax, pos, nodes, *, left_x=-0.04, right_x=1.04, min_vertical_gap=0.055):
    """Label a small subset of nodes with external callouts on fixed left/right columns.

    Each label is anchored at a fixed column in axes-fraction space and receives a
    vertical slot that preserves the node order but enforces a minimum gap between
    labels on the same side.
    """
    nodes = list(nodes)
    if not nodes:
        return ax

    coords = np.asarray([pos[node] for node in nodes], dtype=float)
    centroid_x = coords[:, 0].mean()

    def compute_label_slots(group):
        """Return non-overlapping y-fractions that preserve the ranking of `group`."""
        if not group:
            return []

        ylim = ax.get_ylim()
        y_span = max(ylim[1] - ylim[0], 1e-9)
        desired = [(pos[node][1] - ylim[0]) / y_span for node in group]

        order = sorted(range(len(group)), key=lambda idx: -desired[idx])
        slots = [None] * len(group)
        prev = None
        for idx in order:
            y = desired[idx] if prev is None else min(desired[idx], prev - min_vertical_gap)
            slots[idx] = y
            prev = y

        # Rescale into [0, 1] while preserving the minimum vertical gap: if the
        # stacked slots overflow the axes, shrink uniformly around the midpoint
        # instead of applying two independent shifts (which could re-introduce
        # overlap below `min_vertical_gap` for large subsets).
        lo, hi = min(slots), max(slots)
        if lo < 0.0 or hi > 1.0:
            span = max(hi - lo, 1e-9)
            scale = min(1.0, 1.0 / span)
            mid = 0.5 * (lo + hi)
            slots = [0.5 + (slot - mid) * scale for slot in slots]
        return slots

    left_nodes = [node for node in nodes if pos[node][0] < centroid_x]
    right_nodes = [node for node in nodes if pos[node][0] >= centroid_x]
    left_slots = compute_label_slots(left_nodes)
    right_slots = compute_label_slots(right_nodes)
    connector = CONNECTOR_STYLE.copy()

    def place_labels(group, slots, anchor_x, ha):
        for node, y_frac in zip(group, slots):
            ax.annotate(
                node,
                xy=pos[node],
                xytext=(anchor_x, y_frac),
                textcoords=ax.transAxes,
                ha=ha,
                va="center",
                fontsize=TEXT["direct_label"],
                fontweight="semibold",
                color="#33241F",
                bbox=LABEL_BBOX,
                arrowprops=connector,
                annotation_clip=False,
                zorder=10,
            )

    place_labels(left_nodes, left_slots, left_x, ha="right")
    place_labels(right_nodes, right_slots, right_x, ha="left")
    return ax


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
draw_network_base(ax, got_graph, got_layout, highlight_nodes=top_labeled_nodes)
add_external_callout_labels(ax, got_layout, top_labeled_nodes)
fit_network_axes(ax, got_layout, pad=0.22)

subtitle, external_callout_summary = summarize_label_clutter(
    fig, ax,
    note="The labels are the same 8 characters, moved outward so they no longer fight the dense core.",
)
add_panel_header(ax, "Stronger choice: external callouts", subtitle)
plt.show()


## Final Scorecard: Three Labeling Strategies on the Same Layout

The figures above used the same weighted layout and the same underlying graph, so the only thing that changed between panels was the **labeling strategy**. The scorecard below makes the progression explicit.


In [ ]:
label_scorecard = pd.DataFrame([
    {
        "strategy": "Label every node",
        "labels overlapping another label": f"{label_all_summary['n_overlap']}/{label_all_summary['n_total']}",
        "fraction": round(label_all_summary["fraction"], 2),
    },
    {
        "strategy": "Top-8 labels, placed internally",
        "labels overlapping another label": f"{internal_label_summary['n_overlap']}/{internal_label_summary['n_total']}",
        "fraction": round(internal_label_summary["fraction"], 2),
    },
    {
        "strategy": "Top-8 labels, external callouts",
        "labels overlapping another label": f"{external_callout_summary['n_overlap']}/{external_callout_summary['n_total']}",
        "fraction": round(external_callout_summary["fraction"], 2),
    },
])
display(label_scorecard)


### Reading the Scorecard

Two lessons should stand out:

- **Number of labels dominates on a dense network.** Dropping from "label everything" to a small justified subset cuts the clutter metric by a large factor before any placement strategy is applied.
- **Placement still matters at the subset size.** Once the subset is small, moving labels outward with callouts pushes the metric close to zero, because the remaining collisions were caused by dense-core geometry, not by label count.

The combined rule is the same one the notebook started with: **select first, then place.** Neither step alone produces a readable figure on a network this dense.


## Final Takeaway Checklist

After working through the notebook, a student should be able to answer:

1. Why should weight-aware encodings (layout, edge width) come before any label is added?
2. Why is label overlap a *rendering*-space problem, measured after the figure is drawn?
3. Why is "label every node" almost never the right default on a dense network?
4. What does "selective and justified" labeling mean, and what ranking rules would be appropriate for other networks (e.g. citation counts, PageRank, betweenness)?
5. When does a plain internal label fail, and what is the role of external callouts with connector lines?

The core lesson is simple:

**Select a small, justified subset of labels, place them so they do not fight the dense core, and measure the result the same way you measure any other rendering problem — in points, on the actual figure.**
